# Gold `dim_uc` — V-Energy

Publica a `dim_uc`, o cadastro das UCs no formato do **contrato do portal** (`contracts/tabelas/dim_uc.yaml`, `0.3.0-provisional`): uma linha por UC, com titular, documento mascarado, endereço, município, bairro, classe, alimentador e as flags de GD e de irregularidade confirmada. O portal cruza as outras tabelas com ela pelo `id_uc`: fila de inspeção, KPIs, receita, filtros e Visão 360.

Fontes, todas lidas por versão Delta:
- `v-energy.silver.tb_unidades_consumidoras` (Silver de cadastro): as UCs e os atributos;
- `v-energy.gold.gd_uc` (Gold de GD): `flag_gd`;
- `v-energy.silver.tb_inspecoes` (Silver de inspeções): `flag_irregularidade_confirmada`;
- `v-energy.gold.dim_alimentador` (Gold de continuidade): só para conferir que as duas dimensões saíram da mesma versão do cadastro.

A Gold não descarta UC e copia titular, município, bairro, classe e alimentador sem mudança: as outras Golds usam esses valores da mesma Silver, e o importador refaz a receita por município, bairro e classe a partir desta tabela. O CPF ou CNPJ completo não sai da Silver; a Gold guarda só o documento mascarado. As colunas, os tipos, a ordem, as obrigatórias, os padrões e o domínio vêm do YAML do contrato, lido a cada execução.

O padrão `modo=validar` não grava nada: **Run all** valida as fontes, monta e confere a tabela em memória. Para gravar em `v-energy.gold`, troque o widget para `modo=publicar` antes de executar. Quando o Workflow do Databricks existir (#97), a tarefa vai passar `modo=publicar`, como nas outras Golds. A exportação para o banco do portal é um passo separado (#96).

Regras e dicionário de dados: [`docs/gold/dim_uc.md`](../../docs/gold/dim_uc.md).

In [0]:
"""Transformações e validações da Gold dim_uc."""

import re
import uuid
from functools import reduce
from operator import and_

import yaml
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

VERSAO_REGRAS = 'GOLD_DIM_UC_V1'
CONFIRMADA = 'IRREGULARIDADE CONFIRMADA'
RESULTADOS = [CONFIRMADA, 'SEM IRREGULARIDADE']
# Formatos que o mascarar_documento do backend (backend/app/modules/ucs/service.py) devolve
# sem mascarar de novo: CPF ***.456.789-** e CNPJ **.345.678/0001-**
MASCARA_CPF = r'^\*{3}\.\d{3}\.\d{3}-\*{2}\z'
MASCARA_CNPJ = r'^\*{2}\.\d{3}\.\d{3}/\d{4}-\*{2}\z'
# Colunas de cada fonte que a Gold usa
COLUNAS_FONTES = {
    'cadastro': [
        'id_uc',
        'nome_titular',
        'cpf_cnpj',
        'tipo_documento',
        'logradouro',
        'numero',
        'municipio',
        'bairro',
        'classe',
        'alimentador',
    ],
    'gd_uc': ['id_uc'],
    'inspecoes': ['id_uc', 'resultado'],
    'dim_alimentador': ['alimentador', 'numero_ucs'],
}
# Colunas do contrato copiadas do cadastro sem mudança (contrato: cadastro). As Golds de
# faturas e de continuidade usam os mesmos valores da mesma Silver.
COPIADAS_DO_CADASTRO = {
    'titular': 'nome_titular',
    'municipio': 'municipio',
    'bairro': 'bairro',
    'classe': 'classe',
    'alimentador': 'alimentador',
}
# Tipos do YAML do contrato no Spark; numeric(p,s) vira decimal(p,s)
TIPOS_CONTRATO = {'text': 'string', 'date': 'date', 'integer': 'int', 'boolean': 'boolean'}
# Commits que não mudam dados; o auto compaction e a predictive optimization do Databricks
# podem gravá-los logo depois de uma escrita
OPERACOES_MANUTENCAO = {'OPTIMIZE', 'VACUUM START', 'VACUUM END'}


def _exigir_colunas(df: DataFrame, colunas: list[str], nome: str) -> None:
    ausentes = set(colunas) - set(df.columns)
    if ausentes:
        raise ValueError(f'{nome}: colunas ausentes: {sorted(ausentes)}')


def _contar(df: DataFrame, regras: dict) -> tuple[dict, int]:
    """Conta, numa passada, as linhas e as violações de cada regra (verdadeira = violação)."""
    contagens = (
        df.agg(
            F.count('*').alias('_linhas'),
            *[F.count(F.when(condicao, 1)).alias(regra) for regra, condicao in regras.items()],
        )
        .first()
        .asDict()
    )
    return contagens, contagens.pop('_linhas')


def _falhar_se(contagens: dict, nome: str) -> dict:
    falhas = {regra: n for regra, n in contagens.items() if n}
    if falhas:
        raise ValueError(f'{nome} reprovada: {falhas}')
    return contagens


def _padrao_inteiro(padrao: str) -> str:
    """O pattern do YAML casando o valor inteiro, como o re.fullmatch do importador.

    No Java, o `$` final também aceita uma quebra de linha depois do valor; o `\\z`, não.
    """
    if padrao.endswith('$') and not padrao.endswith(r'\$'):
        return padrao[:-1] + r'\z'
    return padrao


def ler_contrato(caminho) -> dict:
    """Colunas (nome e tipo Spark, na ordem), chave e especificação de cada coluna do YAML."""
    with open(caminho, encoding='utf-8') as arquivo:
        spec = yaml.safe_load(arquivo)
    if spec.get('table') != 'dim_uc':
        raise ValueError(f'{caminho}: o YAML não é o da dim_uc')
    colunas = []
    for nome, coluna in spec['columns'].items():
        # A regra R1 do contrato exige type, nullable e description em toda coluna
        if not isinstance(coluna, dict) or not {'type', 'nullable', 'description'} <= set(coluna):
            raise ValueError(f'{caminho}: coluna {nome} sem type, nullable ou description')
        decimal = re.fullmatch(r'numeric\((\d+),\s*(\d+)\)', coluna['type'])
        if decimal:
            colunas.append((nome, f'decimal({decimal[1]},{decimal[2]})'))
        elif coluna['type'] in TIPOS_CONTRATO:
            colunas.append((nome, TIPOS_CONTRATO[coluna['type']]))
        else:
            raise ValueError(f'{caminho}: tipo {coluna["type"]} de {nome} sem equivalente no Spark')
    return {
        'versao': spec['version'],
        'colunas': colunas,
        'chave': spec['primary_key'],
        'especificacao': spec['columns'],
    }


def regras_do_contrato(contrato: dict) -> dict:
    """O que o importador cobra de cada valor (parse de importador/src/importador/contracts.py)."""
    regras = {}
    for nome, coluna in contrato['especificacao'].items():
        valor = F.col(nome)
        if not coluna['nullable']:
            regras[f'contrato: {nome} nulo'] = valor.isNull()
        if coluna['type'] == 'text':
            regras[f'contrato: {nome} vazio'] = F.trim(valor) == ''
        if coluna.get('pattern'):
            regras[f'contrato: {nome} fora do padrão {coluna["pattern"]}'] = ~valor.rlike(
                _padrao_inteiro(coluna['pattern'])
            )
        if 'enum' in coluna:
            regras[f'contrato: {nome} fora do domínio'] = ~valor.isin(coluna['enum'])
    return regras


def validar_fontes(fontes: dict) -> dict:
    """Premissas das fontes que a Gold usa: grão, documento, endereço e referências ao cadastro."""
    for nome, colunas in COLUNAS_FONTES.items():
        _exigir_colunas(fontes[nome], colunas, f'Fonte {nome}')
    cadastro, gd_uc = fontes['cadastro'], fontes['gd_uc']
    inspecoes, dim_alimentador = fontes['inspecoes'], fontes['dim_alimentador']
    documento = F.col('cpf_cnpj')
    tamanho_coerente = ((F.col('tipo_documento') == 'CPF') & (F.length(documento) == 11)) | (
        (F.col('tipo_documento') == 'CNPJ') & (F.length(documento) == 14)
    )
    contagens, _ = _contar(
        cadastro,
        {
            'cadastro: id_uc nulo': F.col('id_uc').isNull(),
            'cadastro: documento fora de 11 (CPF) ou 14 (CNPJ) dígitos': documento.isNull()
            | ~documento.rlike(r'^(\d{11}|\d{14})\z'),
            'cadastro: tipo_documento diferente do tamanho do documento': ~F.coalesce(
                tamanho_coerente, F.lit(False)
            ),
            'cadastro: logradouro vazio (sem endereço)': F.coalesce(F.trim('logradouro'), F.lit(''))
            == '',
        },
    )
    contagens |= _contar(
        inspecoes,
        {
            'inspeções: resultado fora do domínio': ~F.coalesce(
                F.col('resultado').isin(RESULTADOS), F.lit(False)
            ),
        },
    )[0]
    ids_cadastro = cadastro.select('id_uc')
    contagens |= {
        'cadastro: id_uc repetido (linhas a mais)': cadastro.count()
        - ids_cadastro.distinct().count(),
        'gd_uc: id_uc repetido (linhas a mais)': gd_uc.count()
        - gd_uc.select('id_uc').distinct().count(),
        # O gate R4 do importador reprova UC de gd_uc fora da dim_uc
        'gd_uc: UC ausente do cadastro': gd_uc.join(ids_cadastro, 'id_uc', 'left_anti').count(),
        'inspeções: UC ausente do cadastro': inspecoes.join(
            ids_cadastro, 'id_uc', 'left_anti'
        ).count(),
        'dim_alimentador: alimentador repetido (linhas a mais)': dim_alimentador.count()
        - dim_alimentador.select('alimentador').distinct().count(),
    }
    return _falhar_se(contagens, 'Fontes da dim_uc')


def mascarar_documento(documento) -> F.Column:
    """CPF ***.456.789-** e CNPJ **.345.678/0001-**, como o mascarar_documento do backend."""
    return F.when(
        F.length(documento) == 11,
        F.concat(
            F.lit('***.'),
            F.substring(documento, 4, 3),
            F.lit('.'),
            F.substring(documento, 7, 3),
            F.lit('-**'),
        ),
    ).when(
        F.length(documento) == 14,
        F.concat(
            F.lit('**.'),
            F.substring(documento, 3, 3),
            F.lit('.'),
            F.substring(documento, 6, 3),
            F.lit('/'),
            F.substring(documento, 9, 4),
            F.lit('-**'),
        ),
    )


def _ucs_confirmadas(inspecoes: DataFrame) -> DataFrame:
    return inspecoes.filter(F.col('resultado') == CONFIRMADA).select('id_uc').distinct()


def construir_dim_uc(fontes: dict, contrato: dict) -> DataFrame:
    """Uma linha por UC do cadastro, nas colunas e na ordem do YAML do contrato."""
    gd = fontes['gd_uc'].select('id_uc').distinct().withColumn('_com_gd', F.lit(True))
    confirmadas = _ucs_confirmadas(fontes['inspecoes']).withColumn('_confirmada', F.lit(True))
    dim = (
        fontes['cadastro']
        .join(gd, 'id_uc', 'left')
        .join(confirmadas, 'id_uc', 'left')
        .select(
            'id_uc',
            *[F.col(origem).alias(destino) for destino, origem in COPIADAS_DO_CADASTRO.items()],
            mascarar_documento(F.col('cpf_cnpj')).alias('documento_mascarado'),
            # Sem número, o endereço fica só com o logradouro
            F.concat_ws(', ', 'logradouro', F.when(F.trim('numero') != '', F.col('numero'))).alias(
                'endereco'
            ),
            F.coalesce('_com_gd', F.lit(False)).alias('flag_gd'),
            F.coalesce('_confirmada', F.lit(False)).alias('flag_irregularidade_confirmada'),
        )
    )
    colunas = [c for c, _ in contrato['colunas']]
    faltando = [c for c in colunas if c not in dim.columns]
    if faltando:
        raise ValueError(f'O YAML do contrato pede colunas que a Gold não gera: {faltando}')
    dim = dim.select(*colunas)
    # O comentário de cada coluna no catálogo é a description do YAML
    for coluna in colunas:
        dim = dim.withMetadata(
            coluna, {'comment': contrato['especificacao'][coluna]['description']}
        )
    return dim


def _divergencias_da_flag(dim: DataFrame, flag: str, ids: DataFrame) -> int:
    """UCs com a flag fora de ids, mais ids sem a flag (inclusive fora da dim_uc)."""
    marcadas = dim.filter(F.col(flag)).select('id_uc')
    return (
        marcadas.join(ids, 'id_uc', 'left_anti').count()
        + ids.join(marcadas, 'id_uc', 'left_anti').count()
    )


def validar_gold(dim: DataFrame, fontes: dict, contrato: dict, rotulo: str) -> tuple[dict, int]:
    """Contrato, nenhuma UC descartada, atributos iguais ao cadastro, flags e alimentadores.

    Devolve as violações por regra e as linhas da tabela.
    """
    esquema = [(c.name, c.dataType.simpleString()) for c in dim.schema]
    # Antes das regras por valor, que supõem os tipos do YAML
    if esquema != contrato['colunas']:
        raise ValueError(
            f'dim_uc ({rotulo}) reprovada: colunas, tipos ou ordem diferentes do YAML. '
            f'Esperado {contrato["colunas"]}, gerado {esquema}'
        )
    chave = contrato['chave']
    mascara = F.col('documento_mascarado')
    contagens, linhas = _contar(
        dim.withColumn('_n', F.count('*').over(Window.partitionBy(*chave))),
        {
            f'contrato: chave ({", ".join(chave)}) repetida': F.col('_n') > 1,
            **regras_do_contrato(contrato),
            'documento_mascarado fora das máscaras de CPF e CNPJ do backend': ~(
                mascara.rlike(MASCARA_CPF) | mascara.rlike(MASCARA_CNPJ)
            ),
        },
    )
    contagens['contrato: colunas, tipos ou ordem diferentes do YAML'] = 0

    # Nenhuma UC descartada: a Gold de medidores e as outras conferem o id_uc contra o cadastro
    cadastro = fontes['cadastro']
    contagens['UC do cadastro ausente da dim_uc'] = cadastro.join(dim, 'id_uc', 'left_anti').count()
    contagens['UC da dim_uc ausente do cadastro'] = dim.join(cadastro, 'id_uc', 'left_anti').count()

    # Atributos iguais aos do cadastro; os dígitos visíveis da máscara são os do documento
    g, s = dim.alias('g'), cadastro.alias('s')
    iguais = reduce(
        and_,
        [F.col(f'g.{d}').eqNullSafe(F.col(f's.{o}')) for d, o in COPIADAS_DO_CADASTRO.items()],
    )
    documento, mascara_g = F.col('s.cpf_cnpj'), F.col('g.documento_mascarado')
    digitos = F.regexp_replace(mascara_g, r'\D', '')
    corresponde = (
        (F.length(documento) == 11)
        & mascara_g.rlike(MASCARA_CPF)
        & (digitos == F.substring(documento, 4, 6))
    ) | (
        (F.length(documento) == 14)
        & mascara_g.rlike(MASCARA_CNPJ)
        & (digitos == F.substring(documento, 3, 10))
    )
    contagens |= _contar(
        g.join(s, F.col('g.id_uc') == F.col('s.id_uc')),
        {
            'titular, município, bairro, classe ou alimentador diferente do cadastro': ~iguais,
            'documento_mascarado não corresponde ao documento do cadastro': ~F.coalesce(
                corresponde, F.lit(False)
            ),
        },
    )[0]

    # R4 do importador e flag de irregularidade, nos dois sentidos
    contagens['flag_gd diferente da presença em gd_uc (R4)'] = _divergencias_da_flag(
        dim, 'flag_gd', fontes['gd_uc'].select('id_uc').distinct()
    )
    contagens['flag_irregularidade_confirmada diferente das inspeções confirmadas'] = (
        _divergencias_da_flag(
            dim, 'flag_irregularidade_confirmada', _ucs_confirmadas(fontes['inspecoes'])
        )
    )

    # A dim_alimentador sai do mesmo cadastro: mesmas UCs por alimentador, mesma versão
    dim_alimentador = fontes['dim_alimentador'].select('alimentador', 'numero_ucs')
    contagens['alimentador ausente da dim_alimentador'] = (
        dim.select('alimentador').distinct().join(dim_alimentador, 'alimentador', 'left_anti')
    ).count()
    contagens['UCs por alimentador diferentes da dim_alimentador (outra versão do cadastro)'] = (
        dim.groupBy('alimentador')
        .agg(F.count('*').alias('_ucs'))
        .join(dim_alimentador, 'alimentador', 'full_outer')
        .filter(~F.col('_ucs').eqNullSafe(F.col('numero_ucs')))
        .count()
    )
    return _falhar_se(contagens, f'dim_uc ({rotulo})'), linhas


def _commits_depois(spark, destino: str, anterior) -> list:
    """Commits do destino depois da versão anterior (todos, se a tabela não existia)."""
    return (
        spark.sql(f'DESCRIBE HISTORY {destino}')
        .filter(F.col('version') > (-1 if anterior is None else anterior))
        .select('version', 'operation', 'userMetadata')
        .collect()
    )


def _recuperar(spark, destino: str, anterior, execucao: str) -> None:
    """Volta o destino à versão de antes desta execução, se ninguém mais o gravou nesse meio.

    Depois da versão anterior só pode haver commits desta execução (com o id dela em
    userMetadata) e manutenção que não muda dados. Outra escrita seria descartada por um
    RESTORE às cegas.
    """
    if not spark.catalog.tableExists(destino):
        return
    commits = _commits_depois(spark, destino, anterior)
    de_fora = [
        (c['version'], c['operation'])
        for c in commits
        if c['userMetadata'] != execucao and c['operation'] not in OPERACOES_MANUTENCAO
    ]
    if de_fora:
        raise RuntimeError(
            f'{destino}: outra escrita entrou junto com esta execução (versão anterior '
            f'{anterior}, commits de fora {de_fora}); recupere à mão'
        )
    if not any(c['userMetadata'] == execucao for c in commits):
        # A escrita desta execução não chegou a confirmar uma versão
        return
    if anterior is None:
        spark.sql(f'DROP TABLE IF EXISTS {destino}')
    else:
        spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {anterior}')


def publicar_delta(spark, tabelas: dict, destinos: dict, validar) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha.

    Cada escrita leva o id da execução em userMetadata, que identifica no histórico Delta as
    versões gravadas por ela. Mesmo código da Gold de medidores (#105), até o módulo comum
    de publicação (#139).
    """
    execucao = f'{VERSAO_REGRAS}:{uuid.uuid4()}'
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise ValueError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .option('userMetadata', execucao)
                .saveAsTable(destino)
            )
            # A versão desta escrita, e não a última: um OPTIMIZE automático pode vir depois
            proprias = [
                c['version']
                for c in _commits_depois(spark, destino, anteriores[nome])
                if c['userMetadata'] == execucao
            ]
            if len(proprias) != 1:
                raise ValueError(f'{destino}: esperava 1 versão desta execução, achei {proprias}')
            versao = proprias[0]
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise ValueError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise ValueError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                _recuperar(spark, destino, anteriores[nome], execucao)
            except Exception as falha:
                falhas.append(RuntimeError(f'Falha ao recuperar {destino}: {falha}'))
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [0]:
# ruff: noqa: F821
from pathlib import Path

parametros = {
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'tabela_inspecoes': 'v-energy.silver.tb_inspecoes',
    'tabela_gd_uc': 'v-energy.gold.gd_uc',
    'tabela_dim_alimentador': 'v-energy.gold.dim_alimentador',
    # Relativo à pasta do notebook na pasta Git do repositório
    'contrato_yaml': '../../contracts/tabelas/dim_uc.yaml',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
# O Unity Catalog não diferencia maiúsculas: `Silver` é o schema silver
if config['schema_gold'].split('.')[1].lower() in {'silver', 'bronze', 'default'}:
    raise ValueError('O destino deve ser um schema próprio da Gold')
especificacao = ler_contrato(Path(config['contrato_yaml']))
fontes_lidas = {}
print(
    f'Schema Gold: {schema_gold} | Modo: {config["modo"]} | Regras: {VERSAO_REGRAS} | '
    f'Contrato: {especificacao["versao"]}'
)
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. A dim_uc será gravada em {schema_gold}')

Schema Gold: `v-energy`.`gold` | Modo: publicar | Regras: GOLD_DIM_UC_V1 | Contrato: 0.3.0-provisional
ATENÇÃO: modo=publicar. A dim_uc será gravada em `v-energy`.`gold`


## 1. Leitura consistente das fontes (`VERSION AS OF`) e validação

Fixa a versão Delta atual de cada fonte no início da execução, então a validação, a montagem e a escrita leem exatamente o mesmo snapshot. Se uma fonte estiver em outro catálogo ou schema no seu workspace, informe o nome no widget correspondente.

A `dim_uc`, a `dim_alimentador` (Gold de continuidade) e as Golds de faturas e de medidores precisam ler a **mesma versão** da Silver de cadastro: o importador confere as chaves estrangeiras e refaz a receita por município, bairro e classe a partir da `dim_uc`. O notebook para se as UCs por alimentador não baterem com a `dim_alimentador` publicada.

In [0]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise ValueError(
            f'Tabela não encontrada: {tabela}. '
            f'Publique a fonte correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise ValueError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes_lidas[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
        'timestamp_commit': str(historico['timestamp']),
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


fontes = {
    'cadastro': ler_snapshot('tabela_cadastro'),
    'gd_uc': ler_snapshot('tabela_gd_uc'),
    'inspecoes': ler_snapshot('tabela_inspecoes'),
    'dim_alimentador': ler_snapshot('tabela_dim_alimentador'),
}
regras_fonte = validar_fontes(fontes)
dim_uc = construir_dim_uc(fontes, especificacao)
regras_gold, linhas_gold = validar_gold(dim_uc, fontes, especificacao, 'em memória')

display(
    spark.createDataFrame(
        [(r, n, 'OK') for r, n in {**regras_fonte, **regras_gold}.items()],
        'regra string, violacoes long, status string',
    )
)
print(f'[OK] dim_uc: {linhas_gold:,} UCs validadas em memória')

tabela_cadastro: `v-energy`.`silver`.`tb_unidades_consumidoras` (versão 43)
tabela_gd_uc: `v-energy`.`gold`.`gd_uc` (versão 3)
tabela_inspecoes: `v-energy`.`silver`.`tb_inspecoes` (versão 2)
tabela_dim_alimentador: `v-energy`.`gold`.`dim_alimentador` (versão 0)


regra,violacoes,status
cadastro: id_uc nulo,0,OK
cadastro: documento fora de 11 (CPF) ou 14 (CNPJ) dígitos,0,OK
cadastro: tipo_documento diferente do tamanho do documento,0,OK
cadastro: logradouro vazio (sem endereço),0,OK
inspeções: resultado fora do domínio,0,OK
cadastro: id_uc repetido (linhas a mais),0,OK
gd_uc: id_uc repetido (linhas a mais),0,OK
gd_uc: UC ausente do cadastro,0,OK
inspeções: UC ausente do cadastro,0,OK
dim_alimentador: alimentador repetido (linhas a mais),0,OK


[OK] dim_uc: 45,000 UCs validadas em memória


## 2. Verificação das perguntas de negócio do case

Confere direto na tabela montada. Titular e endereço ficam fora das saídas salvas, porque são dados pessoais:
- **Visão 360 (Camila Barreto, Atendimento):** de quem é a UC e onde fica? O cabeçalho mostra titular, documento mascarado, classe, endereço e alimentador.
- **Filtros do portal e da IA:** os municípios, bairros, classes e alimentadores que o portal lista vêm desta tabela.
- **Grupos de UC (Henrique Duarte, Comercial; Rafael Coutinho, Perdas):** quantas UCs têm GD e quantas já tiveram irregularidade confirmada, no retrato de hoje.

In [0]:
# ruff: noqa: F821
print('=== 1) Visão 360: cabeçalho da UC100015, sem titular e endereço ===')
display(
    dim_uc.filter(F.col('id_uc') == 'UC100015').select(
        'id_uc',
        'documento_mascarado',
        'classe',
        'municipio',
        'bairro',
        'alimentador',
        'flag_gd',
        'flag_irregularidade_confirmada',
    )
)

print('=== 2) Recortes dos filtros do portal ===')
display(
    dim_uc.agg(
        F.count('*').alias('ucs'),
        F.countDistinct('municipio').alias('municipios'),
        F.countDistinct('bairro').alias('bairros'),
        F.countDistinct('municipio', 'bairro').alias('pares_municipio_bairro'),
        F.countDistinct('classe').alias('classes'),
        F.countDistinct('alimentador').alias('alimentadores'),
    )
)

sem_numero = fontes['cadastro'].filter(F.coalesce(F.trim('numero'), F.lit('')) == '').count()
print(f'UCs sem número (endereço só com o logradouro): {sem_numero:,}')

print(
    '=== 3) Nomes de bairro em mais de um município: o filtro por bairro precisa do município ==='
)
display(
    dim_uc.groupBy('bairro')
    .agg(F.countDistinct('municipio').alias('municipios'))
    .filter(F.col('municipios') > 1)
    .orderBy(F.desc('municipios'), 'bairro')
)

print('=== 4) UCs por classe, com GD e com irregularidade confirmada ===')
display(
    dim_uc.groupBy('classe')
    .agg(
        F.count('*').alias('ucs'),
        F.count_if('flag_gd').alias('com_gd'),
        F.count_if('flag_irregularidade_confirmada').alias('com_irregularidade_confirmada'),
        F.count_if(F.col('flag_gd') & F.col('flag_irregularidade_confirmada')).alias('com_as_duas'),
    )
    .orderBy(F.desc('ucs'))
)

print('=== 5) UCs por município ===')
display(
    dim_uc.groupBy('municipio')
    .agg(F.count('*').alias('ucs'), F.countDistinct('bairro').alias('bairros'))
    .orderBy(F.desc('ucs'), 'municipio')
)

print('=== 6) Documento mascarado por tipo ===')
display(
    dim_uc.groupBy(
        F.when(F.col('documento_mascarado').rlike(MASCARA_CPF), 'CPF')
        .when(F.col('documento_mascarado').rlike(MASCARA_CNPJ), 'CNPJ')
        .alias('tipo_documento')
    )
    .count()
    .orderBy('tipo_documento')
)

=== 1) Visão 360: cabeçalho da UC100015, sem titular e endereço ===


id_uc,documento_mascarado,classe,municipio,bairro,alimentador,flag_gd,flag_irregularidade_confirmada
UC100015,***.994.720-**,Residencial,Jaboatão dos Guararapes,Curado,ALM-JAB-004,true,false


=== 2) Recortes dos filtros do portal ===


ucs,municipios,bairros,pares_municipio_bairro,classes,alimentadores
45000,14,272,297,5,122


UCs sem número (endereço só com o logradouro): 0
=== 3) Nomes de bairro em mais de um município: o filtro por bairro precisa do município ===


bairro,municipios
Centro,10
Bela Vista,4
COHAB,2
Curado,2
Distrito Industrial,2
Fragoso,2
Jaguaribe,2
Mangabeira,2
Mangueira,2
Nossa Senhora da Conceição,2


... (2 outputs left out)

=== 5) UCs por município ===


municipio,ucs,bairros
Recife,17995,86
Jaboatão dos Guararapes,7800,26
Olinda,4212,24
Paulista,4024,21
Cabo de Santo Agostinho,2513,16
Camaragibe,1783,14
Igarassu,1400,15
São Lourenço da Mata,1398,8
Ipojuca,1176,26
Abreu e Lima,1165,10


=== 6) Documento mascarado por tipo ===


tipo_documento,count
CNPJ,6008
CPF,38992


## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da escrita, registra a versão atual da tabela de destino. A escrita leva o id da execução no `userMetadata` do commit Delta. Depois de gravar, confere o schema e o conteúdo gravados e repete todas as regras sobre a tabela gravada. Se algo falhar, restaura o destino com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga), desde que, depois da versão registrada, só haja commits desta execução e de manutenção (`OPTIMIZE`, `VACUUM`).

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise ValueError('Parâmetros mudaram durante a execução. Execute novamente desde o início.')
    tabelas = {'dim_uc': dim_uc}
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes_lidas.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise ValueError(f'A identidade da tabela fonte mudou: {chave}')
    # Comparação sem maiúsculas, como o Unity Catalog resolve os nomes
    if {d.lower() for d in destinos.values()} & {
        qualificar(config[c]).lower() for c in fontes_lidas
    }:
        raise ValueError('A tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_gold(gravadas['dim_uc'], fontes, especificacao, 'tabela gravada'),
    )
    for nome, info in versoes_gold.items():
        print(
            f'Publicada: {info["tabela"]} (versão {info["versao_delta"]}, '
            f'com o id desta execução no userMetadata)'
        )

Publicada: `v-energy`.`gold`.`dim_uc` (versão 0, com o id desta execução no userMetadata)
